In [ ]:
pip install matplotlib

In [ ]:
import os
import json
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

def verify_cropped_dataset(base_dir, json_name="cropped_annotations.json", num_samples=3):
    json_path = os.path.join(base_dir, json_name)
    image_dir = os.path.join(base_dir, "images")
    
    # Load COCO JSON
    with open(json_path, 'r') as f:
        coco = json.load(f)
        
    categories = {cat['id']: cat['name'] for cat in coco.get('categories', [])}
    
    # Group annotations by image ID
    img_to_anns = {}
    for ann in coco.get('annotations', []):
        img_to_anns.setdefault(ann['image_id'], []).append(ann)
        
    # Randomly select a few images
    all_images = coco.get('images', [])
    if not all_images:
        print("No images found in JSON!")
        return
        
    sample_images = random.sample(all_images, min(num_samples, len(all_images)))
    
    plt.figure(figsize=(15, 5 * num_samples))
    
    for i, img_info in enumerate(sample_images):
        img_path = os.path.join(image_dir, img_info['file_name'])
        
        # Load image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Could not load {img_path}")
            continue
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Create a copy for drawing solid lines and an overlay for transparency
        img_with_polys = img.copy()
        overlay = img.copy()
        
        # Draw segmentations
        anns = img_to_anns.get(img_info['id'], [])
        for ann in anns:
            cat_name = categories.get(ann['category_id'], "Unknown")
            
            # Choose color based on category (Red for anode, Blue for cathode)
            color = (255, 0, 0) if cat_name.lower() == 'anode' else (0, 0, 255)
            
            if 'segmentation' in ann and ann['segmentation']:
                for poly in ann['segmentation']:
                    # Convert flat COCO list [x1, y1, x2, y2...] into numpy array of shape (N, 1, 2)
                    pts = np.array(poly, np.int32).reshape((-1, 1, 2))
                    
                    # Fill polygon on the overlay (for transparency)
                    cv2.fillPoly(overlay, [pts], color)
                    
                    # Draw solid boundary line on the main image
                    cv2.polylines(img_with_polys, [pts], isClosed=True, color=color, thickness=2)
                    
                    # Draw Label at the first point of the polygon
                    text_x, text_y = pts[0][0][0], pts[0][0][1]
                    cv2.putText(img_with_polys, cat_name, (text_x, max(15, text_y - 5)), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                                
        # Apply alpha blending for semi-transparent masks
        alpha = 0.4
        cv2.addWeighted(overlay, alpha, img_with_polys, 1 - alpha, 0, img_with_polys)

        # Plot Original vs Labeled
        plt.subplot(num_samples, 2, 2*i + 1)
        plt.imshow(img)
        plt.title(f"Original Crop: {img_info['file_name']}")
        plt.axis("off")
        
        plt.subplot(num_samples, 2, 2*i + 2)
        plt.imshow(img_with_polys)
        plt.title(f"Labeled ({len(anns)} objects)")
        plt.axis("off")
        
    plt.tight_layout()
    plt.show()

# Run the verification
BASE_DIR = r"E:\Prithu\cropped_dataset"
verify_cropped_dataset(BASE_DIR, num_samples=3)

In [ ]:
import os
import json
import torch
import cv2
import numpy as np
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import albumentations as A
from torch.utils.data import Dataset
from transformers import (
    Mask2FormerImageProcessor,
    Mask2FormerForUniversalSegmentation,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback
)

# 1. Augmentations (Horizontal flips & subtle spatial distortion)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(translate_percent=0.05, scale=(0.95, 1.05), rotate=0, p=0.5),
    A.ElasticTransform(p=0.2, alpha=1, sigma=50),
])

# 2. COCO Dataset Loader (Parses COCO JSON directly)
class COCOWireDataset(Dataset):
    def __init__(self, json_path, image_dir, processor, transforms=None):
        self.image_dir = image_dir
        self.processor = processor
        self.transforms = transforms

        # Load COCO JSON
        with open(json_path, 'r') as f:
            self.coco = json.load(f)

        # Build category mapping (e.g., 'anode' -> 0, 'cathode' -> 1)
        self.categories = self.coco.get('categories', [])
        self.cat_id_to_sem_id = {cat['id']: idx for idx, cat in enumerate(self.categories)}
        
        # Map images to their annotations
        self.images = self.coco.get('images', [])
        self.img_to_anns = {}
        for ann in self.coco.get('annotations', []):
            self.img_to_anns.setdefault(ann['image_id'], []).append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']
        file_name = img_info['file_name']
        image_path = os.path.join(self.image_dir, file_name)

        # Load image
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Image not found: {image_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        # Generate 2D Instance Mask (Background is 255)
        anns = self.img_to_anns.get(img_id, [])
        instance_mask = np.full((h, w), 255, dtype=np.int32)
        instance_id_to_semantic_id = {}

        for inst_idx, ann in enumerate(anns, start=1):
            # ---------------------------------------------------------
            # NEW: Draw precise Polygons instead of Bounding Boxes
            # ---------------------------------------------------------
            if 'segmentation' in ann and ann['segmentation']:
                for poly in ann['segmentation']:
                    # Convert [x1, y1, x2, y2...] to numpy array of shape (N, 2)
                    pts = np.array(poly, np.int32).reshape((-1, 2))
                    # Paint the exact polygon shape onto the mask with the instance ID
                    cv2.fillPoly(instance_mask, [pts], inst_idx)
            else:
                # Fallback to bounding box ONLY if polygon data is missing
                bbox = ann['bbox'] 
                x, y, bw, bh = [int(v) for v in bbox]
                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(w, x + bw), min(h, y + bh)
                instance_mask[y1:y2, x1:x2] = inst_idx
            # ---------------------------------------------------------

            # Map category to 0 (anode) or 1 (cathode)
            cat_id = ann['category_id']
            instance_id_to_semantic_id[inst_idx] = self.cat_id_to_sem_id[cat_id]

        # Apply Albumentations
        # Apply Albumentations
        if self.transforms:
            augmented = self.transforms(image=image, mask=instance_mask)
            image = augmented['image']
            instance_mask = augmented['mask']
            
            # FIX: Albumentations pads empty borders with 0. Change them back to background (255)
            instance_mask[instance_mask == 0] = 255

        # Build ground-truth semantic map for validation metrics
        semantic_target = np.zeros((h, w), dtype=np.uint8)
        for inst_id, sem_id in instance_id_to_semantic_id.items():
            semantic_target[instance_mask == inst_id] = sem_id + 1

        return image, instance_mask, instance_id_to_semantic_id, semantic_target


def collate_fn(batch):
    images = [item[0] for item in batch]
    segmentation_maps = [item[1] for item in batch]
    instance_id_to_semantic_id = [item[2] for item in batch]

    inputs = processor(
        images=images,
        segmentation_maps=segmentation_maps,
        instance_id_to_semantic_id=instance_id_to_semantic_id,
        return_tensors="pt",
    )
    return inputs


# 3. Custom Evaluation Callback (For Epoch Metric Logging)
class ThresholdEvaluationCallback(TrainerCallback):
    def __init__(self, val_dataset, processor, anode_thresh=0.50, cathode_thresh=0.50):
        self.val_dataset = val_dataset
        self.processor = processor
        self.anode_thresh = anode_thresh
        self.cathode_thresh = cathode_thresh

    def on_evaluate(self, args, state, control, model, **kwargs):
        model.eval()
        device = args.device

        ious_anode, dices_anode = [], []
        ious_cathode, dices_cathode = [], []

        for idx in range(len(self.val_dataset)):
            image, _, _, semantic_target = self.val_dataset[idx]
            inputs = self.processor(images=[image], return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model(**inputs)
                class_logits = outputs.class_queries_logits
                mask_logits = outputs.masks_queries_logits

                class_probs = class_logits.softmax(dim=-1)[..., :-1]
                mask_probs = mask_logits.sigmoid()

                # Combine query predictions into class spatial probabilities
                semantic_probs = torch.einsum("bqc, bqhw -> bchw", class_probs, mask_probs)

                # FIX: Resize model's 96x96 output back to the 540x540 image size
                semantic_probs = torch.nn.functional.interpolate(
                    semantic_probs, 
                    size=semantic_target.shape, 
                    mode="bilinear", 
                    align_corners=False
                )

                pred_anode = (semantic_probs[0, 0] > self.anode_thresh).cpu().numpy()
                pred_cathode = (semantic_probs[0, 1] > self.cathode_thresh).cpu().numpy()

            target_anode = (semantic_target == 1)
            target_cathode = (semantic_target == 2)

            def calc_metrics(pred, target):
                inter = np.logical_and(pred, target).sum()
                union = np.logical_or(pred, target).sum()
                iou = inter / union if union > 0 else np.nan
                dice = 2 * inter / (pred.sum() + target.sum()) if (pred.sum() + target.sum()) > 0 else np.nan
                return iou, dice

            iou_a, dice_a = calc_metrics(pred_anode, target_anode)
            iou_c, dice_c = calc_metrics(pred_cathode, target_cathode)

            if not np.isnan(iou_a): ious_anode.append(iou_a)
            if not np.isnan(dice_a): dices_anode.append(dice_a)
            if not np.isnan(iou_c): ious_cathode.append(iou_c)
            if not np.isnan(dice_c): dices_cathode.append(dice_c)

        print(f"\n--- Epoch {state.epoch} Validation Metrics ---")
        print(f"Anode   (Thresh {self.anode_thresh:.2f}) | IoU: {np.mean(ious_anode):.4f} | Dice: {np.mean(dices_anode):.4f}")
        print(f"Cathode (Thresh {self.cathode_thresh:.2f}) | IoU: {np.mean(ious_cathode):.4f} | Dice: {np.mean(dices_cathode):.4f}")
        print("-------------------------------------------\n")


if __name__ == "__main__":
    # --- Paths for your Cropped Dataset ---
    BASE_DIR = r"E:\Prithu\cropped_dataset"
    JSON_PATH = os.path.join(BASE_DIR, "cropped_annotations.json")
    IMAGE_DIR = os.path.join(BASE_DIR, "images")
    
    MODEL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"
    ID2LABEL = {0: "anode", 1: "cathode"}

    processor = Mask2FormerImageProcessor.from_pretrained(
        MODEL_CHECKPOINT, 
        ignore_index=255, 
        reduce_labels=False 
    )
    model = Mask2FormerForUniversalSegmentation.from_pretrained(
        MODEL_CHECKPOINT, id2label=ID2LABEL, ignore_mismatched_sizes=True
    )

    # Initialize Dataset
    full_dataset = COCOWireDataset(
        json_path=JSON_PATH, 
        image_dir=IMAGE_DIR, 
        processor=processor, 
        transforms=train_transform
    )

    # Split dataset into Train (80%) and Validation (20%)
    val_size = int(0.2 * len(full_dataset))
    train_size = len(full_dataset) - val_size
    # Fix random seed for consistent train/val splits across runs
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size], generator=generator)

    # Optimizer: Layer-wise Learning Rate Decay
    backbone_params = [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad]
    decoder_params = [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": 1e-5},
        {"params": decoder_params, "lr": 1e-4},
    ], weight_decay=0.05)

    # Training Configuration
    training_args = TrainingArguments(
        output_dir=os.path.join(BASE_DIR, "mask2former_model"),
        num_train_epochs=400,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        per_device_train_batch_size=24,
        per_device_eval_batch_size=24,
        dataloader_num_workers=0,
        fp16=True,
        remove_unused_columns=False,
    )

    early_stop = EarlyStoppingCallback(early_stopping_patience=50)
    custom_metrics = ThresholdEvaluationCallback(
        val_dataset=val_dataset,
        processor=processor,
        anode_thresh=0.50,
        cathode_thresh=0.50
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate_fn,
        optimizers=(optimizer, None),
        callbacks=[early_stop, custom_metrics]
    )

    print(f"Loaded {len(full_dataset)} images from {BASE_DIR}")
    print("Starting Training...")
    trainer.train()

In [ ]:
import os
import random
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

# 1. Force PyTorch to use your RTX 6000 (GPU 1)
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# 2. Paths
BASE_DIR = r"E:\Prithu\cropped_dataset"
IMAGE_DIR = os.path.join(BASE_DIR, "images")
MODEL_DIR = os.path.join(BASE_DIR, "mask2former_model")

# 3. Load the Fine-Tuned Model and Processor
# 3. Find the Latest Checkpoint
checkpoint_folders = [f for f in os.listdir(MODEL_DIR) if f.startswith("checkpoint-")]
# Sort numerically by the checkpoint number
checkpoint_folders.sort(key=lambda x: int(x.split("-")[-1]))
LATEST_CHECKPOINT = os.path.join(MODEL_DIR, checkpoint_folders[-1])

print(f"Loading model from latest checkpoint: {LATEST_CHECKPOINT}")

# Pull the processor configuration from the original base model
ORIGINAL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"
processor = Mask2FormerImageProcessor.from_pretrained(
    ORIGINAL_CHECKPOINT, 
    ignore_index=255, 
    reduce_labels=False
)

# Load your custom fine-tuned weights from the checkpoint folder
model = Mask2FormerForUniversalSegmentation.from_pretrained(LATEST_CHECKPOINT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print(f"Model successfully loaded to {device}!")



def visualize_predictions(image_dir, num_samples=3, threshold=0.40):
    # Get all image files
    all_images = [f for f in os.listdir(image_dir) if f.endswith('.jpg') or f.endswith('.png')]
    if not all_images:
        print("No images found in", image_dir)
        return
        
    sample_files = random.sample(all_images, min(num_samples, len(all_images)))
    
    plt.figure(figsize=(15, 6 * num_samples))
    
    for i, filename in enumerate(sample_files):
        img_path = os.path.join(image_dir, filename)
        
        # Load and prep image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        target_h, target_w = image.shape[:2]
        
        # 4. Run Inference
        inputs = processor(images=[image], return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            
            # Extract logits
            class_logits = outputs.class_queries_logits
            mask_logits = outputs.masks_queries_logits

            # Convert to probabilities
            class_probs = class_logits.softmax(dim=-1)[..., :-1] # Drop the "background" class
            mask_probs = mask_logits.sigmoid()

            # Multiply class confidence by spatial mask probability
            semantic_probs = torch.einsum("bqc, bqhw -> bchw", class_probs, mask_probs)

            # Resize the model's small output map back to 540x540
            semantic_probs = torch.nn.functional.interpolate(
                semantic_probs, 
                size=(target_h, target_w), 
                mode="bilinear", 
                align_corners=False
            )
            
            # Apply our threshold to create boolean masks
            pred_anode = (semantic_probs[0, 0] > threshold).cpu().numpy()
            pred_cathode = (semantic_probs[0, 1] > threshold).cpu().numpy()
            
        # 5. Visualization setup (Alpha Blending)
        overlay = image.copy()
        
        # Color the pixels: Anode = Red, Cathode = Blue
        overlay[pred_anode] = [255, 0, 0]
        overlay[pred_cathode] = [0, 0, 255]
        
        # Blend the colored mask over the original image
        alpha = 0.45
        blended = cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0)
        
        # Plot Original
        plt.subplot(num_samples, 2, 2*i + 1)
        plt.imshow(image)
        plt.title(f"Original: {filename}")
        plt.axis("off")
        
        # Plot Prediction
        plt.subplot(num_samples, 2, 2*i + 2)
        plt.imshow(blended)
        plt.title(f"Prediction (Threshold: {threshold})\nRed: Anode | Blue: Cathode")
        plt.axis("off")
        
    plt.tight_layout()
    plt.show()

# Run the visualizer!
visualize_predictions(IMAGE_DIR, num_samples=4, threshold=0.40)

In [ ]:
import os
import cv2
import torch
import numpy as np
import math  
import matplotlib.pyplot as plt

# Replace torchvision with Transformers
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

# =========================================================
# CONFIG
# =========================================================

BASE_DIR = r"E:\Prithu\cropped_dataset"
# Point this to your saved Mask2Former root directory
MODEL_DIR = os.path.join(BASE_DIR, "mask2former_model")

# DIRECTORY CONTAINING ALL YOUR LARGE IMAGES
INPUT_DIR = r"E:\Prithu\Anode\New folder"
OUTPUT_DIR = os.path.join(BASE_DIR, "predictions_large")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Updated to match Mask2Former training sizes
CROP_SIZE = 540
OVERLAP = 150
STRIDE = CROP_SIZE - OVERLAP

SCORE_THRESHOLD = 0.40 # Using the 0.40 threshold that worked well for wires
MICRONS_PER_PIXEL = 1.72633

# Mapped explicitly to 1 (Anode) and 3 (Cathode)
CLASS_MAPPING = {
    1: {"name": "Anode Wire", "color": (255, 0, 0), "alpha": 0.6, "order": 1},      # Red
    3: {"name": "Cathode Wire", "color": (0, 0, 255), "alpha": 0.6, "order": 2},    # Blue
}

# =========================================================
# MODEL (MASK2FORMER)
# =========================================================

def load_model(model_dir, device):
    # 1. Find the Latest Checkpoint automatically
    checkpoint_folders = [f for f in os.listdir(model_dir) if f.startswith("checkpoint-")]
    if checkpoint_folders:
        checkpoint_folders.sort(key=lambda x: int(x.split("-")[-1]))
        latest_ckpt = os.path.join(model_dir, checkpoint_folders[-1])
    else:
        latest_ckpt = model_dir # Fallback if weights are in the root folder
        
    print(f"Loading model from: {latest_ckpt}")

    # 2. Pull processor config from base model
    ORIGINAL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"
    processor = Mask2FormerImageProcessor.from_pretrained(
        ORIGINAL_CHECKPOINT, 
        ignore_index=255, 
        reduce_labels=False
    )
    
    # 3. Load fine-tuned weights
    model = Mask2FormerForUniversalSegmentation.from_pretrained(latest_ckpt)
    model.to(device)
    model.eval()
    
    return processor, model

# =========================================================
# CLEAN OVERLAP RULES (UNTOUCHED)
# =========================================================

def clean_masks(masks):
    anode_wire = masks[1]
    cath_wire = masks[3]

    overlap_wires = cath_wire & anode_wire
    cath_wire[overlap_wires] = False
    anode_wire[overlap_wires] = False

    return masks

# =========================================================
# DISTANCE (UPDATED WITH LEFT/RIGHT LOGIC)
# =========================================================

def compute_distance_and_lines(masks): 
    mask_anode_wire = masks[1].astype(np.uint8) * 255     
    mask_cath_wire = masks[3].astype(np.uint8) * 255      

    distances_px = []
    traced_paths = [] 
    
    img_height, img_width = mask_anode_wire.shape

    # STEP 1: Find Absolute Top of Cathode Wires
    num_c, labels, stats, _ = cv2.connectedComponentsWithStats(mask_cath_wire)
    cathodes = []

    for i in range(1, num_c):
        x, y, w, h, area = stats[i]
        
        if area < 100 or y < img_height / 3: 
            continue
            
        local_ys, local_xs = np.where(labels == i)
        top_idx = np.argmin(local_ys)
        cx = local_xs[top_idx]
        cy = local_ys[top_idx]
        
        cathodes.append({'x': cx, 'y': cy})
        
    cathodes.sort(key=lambda c: c['x'])

    print("\n" + "="*50)
    print(f" 🛠️ METROLOGY: Tracing {len(cathodes)} Cathode Wires Left-to-Right")
    print("="*50)

    for i, cat in enumerate(cathodes):
        cx, cy = cat['x'], cat['y']

        # NEW LOGIC: First cathode (index 0) scans Left (-1) and Right (1). Others scan only Right (1).
        directions = [-1, 1] if i == 0 else [1]

        for d in directions:
            dir_name = "Left" if d == -1 else "Right"
            start_x, start_y = None, None

            # STEP 2: STRICT HORIZONTAL SCAN
            scan_range = range(cx, min(img_width, cx + 1500)) if d == 1 else range(cx, max(-1, cx - 1500), -1)

            for x_step in scan_range:
                y_min_search = max(0, cy - 5)
                y_max_search = min(img_height, cy + 6)
                
                col = mask_anode_wire[y_min_search:y_max_search, x_step]
                
                if np.any(col > 0):
                    local_ys = np.where(col > 0)[0]
                    actual_ys = local_ys + y_min_search
                    start_x = x_step
                    start_y = actual_ys[np.argmin(np.abs(actual_ys - cy))]
                    break

            if start_x is None:
                print(f"[{i+1}] FAILED: No Anode wire found straight to the {dir_name} of ({cx}, {cy}).")
                continue

            curr_x, curr_y = start_x, start_y 
            path_distance = 0.0 
            current_path = [(cx, cy), (curr_x, curr_y)] 
            
            # STEP 3: STRICT FORWARD (UPWARD) CLIMBER
            while True:
                next_y = curr_y - 1
                if next_y < 0:
                    break 
                
                x_min = max(0, curr_x - 3)
                x_max = min(img_width - 1, curr_x + 3)
                
                row_above = mask_anode_wire[next_y, x_min:x_max+1]
                
                if np.any(row_above > 0):
                    xs = np.where(row_above > 0)[0]
                    next_x = x_min + int(np.mean(xs))
                    
                    path_distance += math.hypot(next_x - curr_x, next_y - curr_y)
                    curr_x = next_x
                    curr_y = next_y
                    current_path.append((curr_x, curr_y))
                else:
                    gap_found = False
                    for jump in range(2, 100): 
                        jump_y = curr_y - jump
                        if jump_y < 0: break
                        
                        jx_min = max(0, curr_x - 10)
                        jx_max = min(img_width - 1, curr_x + 10)
                        
                        gap_row = mask_anode_wire[jump_y, jx_min:jx_max+1]
                        if np.any(gap_row > 0):
                            xs = np.where(gap_row > 0)[0]
                            next_x = jx_min + int(np.mean(xs))
                            
                            path_distance += math.hypot(next_x - curr_x, jump_y - curr_y)
                            curr_x = next_x
                            curr_y = jump_y
                            current_path.append((curr_x, curr_y))
                            gap_found = True
                            break
                            
                    if not gap_found:
                        break 

            # STEP 4: RECORD DISTANCE
            individual_um = path_distance * MICRONS_PER_PIXEL 
            distances_px.append(path_distance)
            traced_paths.append((current_path, individual_um)) 
            print(f"[{i+1}] SUCCESS: Reached Top of Anode Wire ({dir_name}). Traced {individual_um:.2f}um.")

    if len(distances_px) == 0:
        return None, None

    avg_px = np.mean(distances_px)
    avg_um = avg_px * MICRONS_PER_PIXEL  

    return (avg_px, avg_um), traced_paths

# =========================================================
# MAIN (UPDATED WITH BATCH PROCESSING LOOP)
# =========================================================

def main():
    os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("Initializing Model...")
    processor, model = load_model(MODEL_DIR, device)

    # Get all image files in the directory
    valid_extensions = ('.tif', '.tiff', '.png', '.jpg', '.jpeg')
    image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)]
    
    if not image_files:
        print(f"No valid images found in {INPUT_DIR}")
        return

    print(f"Found {len(image_files)} images to process.")

    # Loop through each image
    for img_name in image_files:
        print(f"\n{'='*60}\nProcessing Image: {img_name}\n{'='*60}")
        img_path = os.path.join(INPUT_DIR, img_name)
        
        img = cv2.imread(img_path)
        if img is None:
            print(f"Error loading {img_name}, skipping...")
            continue
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        # Initialize global masks fresh for EVERY image
        global_masks = { i: np.zeros((H, W), dtype=bool) for i in CLASS_MAPPING }

        # =====================================================
        # INFERENCE
        # =====================================================
        with torch.no_grad():
            for y in range(0, H, STRIDE):
                for x in range(0, W, STRIDE):
                    y2 = min(y + CROP_SIZE, H)
                    x2 = min(x + CROP_SIZE, W)

                    crop = img[y:y2, x:x2]
                    ch, cw = crop.shape[:2]

                    padded = np.zeros((CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8)
                    padded[:ch, :cw] = crop

                    inputs = processor(images=[padded], return_tensors="pt").to(device)
                    outputs = model(**inputs)

                    class_logits = outputs.class_queries_logits
                    mask_logits = outputs.masks_queries_logits

                    class_probs = class_logits.softmax(dim=-1)[..., :-1]
                    mask_probs = mask_logits.sigmoid()

                    semantic_probs = torch.einsum("bqc, bqhw -> bchw", class_probs, mask_probs)[0]

                    semantic_probs = torch.nn.functional.interpolate(
                        semantic_probs.unsqueeze(0), 
                        size=(CROP_SIZE, CROP_SIZE), 
                        mode="bilinear", 
                        align_corners=False
                    )[0]

                    pred_anode = (semantic_probs[0] > SCORE_THRESHOLD).cpu().numpy()
                    pred_cathode = (semantic_probs[1] > SCORE_THRESHOLD).cpu().numpy()

                    global_masks[1][y:y2, x:x2] |= pred_anode[:ch, :cw]
                    global_masks[3][y:y2, x:x2] |= pred_cathode[:ch, :cw]

        # =====================================================
        # RENDER SEGMENTATION
        # =====================================================
        vis = img.copy()

        for cid, info in sorted(CLASS_MAPPING.items(), key=lambda x: x[1]["order"]):
            m = global_masks[cid]
            if np.any(m):
                overlay = vis.copy()
                overlay[m] = info["color"]
                vis = cv2.addWeighted(overlay, info["alpha"], vis, 1 - info["alpha"], 0)

        # =====================================================
        # CLEAN MASKS & CALCULATE DISTANCE
        # =====================================================
        cleaned = clean_masks(global_masks)
        result, traced_paths = compute_distance_and_lines(cleaned)

        if result is not None:
            px, avg_um = result
            print(f"\nOVERALL Average Distance for {img_name}: {avg_um:.2f} µm")

            for path_data in traced_paths:
                path_coords = path_data[0]
                specific_um = path_data[1] 

                jump_pts = np.array(path_coords[:2], np.int32).reshape((-1, 1, 2))
                cv2.polylines(vis, [jump_pts], False, (0, 255, 255), 3, cv2.LINE_AA) 
                
                trace_pts = np.array(path_coords[1:], np.int32).reshape((-1, 1, 2))
                cv2.polylines(vis, [trace_pts], False, (255, 255, 255), 3, cv2.LINE_AA)

                top_x, top_y = path_coords[-1]
                cv2.putText(
                    vis, f"{specific_um:.1f}um", (top_x + 15, top_y + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3, cv2.LINE_AA
                )
        else:
            print(f"\n[WARNING] No distances could be calculated for {img_name}.")

        # =====================================================
        # SAVE FULL RESOLUTION
        # =====================================================
        out_path = os.path.join(OUTPUT_DIR, f"merged_{img_name}")
        cv2.imwrite(out_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import cv2
import torch
import numpy as np
import math  
import matplotlib.pyplot as plt

# Replace torchvision with Transformers
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

# =========================================================
# CONFIG
# =========================================================

BASE_DIR = r"E:\Prithu\cropped_dataset"
# Point this to your saved Mask2Former root directory
MODEL_DIR = os.path.join(BASE_DIR, "mask2former_model")

# Change this to whatever large test image you want to run on
IMAGE_PATH = r"E:\Prithu\Anode\New folder\250115_HS24020-01_R1_Kachel_200x_Messungen-01.tif"
OUTPUT_DIR = os.path.join(BASE_DIR, "predictions_large")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Updated to match Mask2Former training sizes
CROP_SIZE = 540
OVERLAP = 150
STRIDE = CROP_SIZE - OVERLAP

SCORE_THRESHOLD = 0.40 # Using the 0.40 threshold that worked well for wires
MICRONS_PER_PIXEL = 1.72633

# Mapped explicitly to 1 (Anode) and 3 (Cathode) so your distance logic remains untouched!
CLASS_MAPPING = {
    1: {"name": "Anode Wire", "color": (255, 0, 0), "alpha": 0.6, "order": 1},      # Red
    3: {"name": "Cathode Wire", "color": (0, 0, 255), "alpha": 0.6, "order": 2},    # Blue
}

# =========================================================
# MODEL (MASK2FORMER)
# =========================================================

def load_model(model_dir, device):
    # 1. Find the Latest Checkpoint automatically
    checkpoint_folders = [f for f in os.listdir(model_dir) if f.startswith("checkpoint-")]
    if checkpoint_folders:
        checkpoint_folders.sort(key=lambda x: int(x.split("-")[-1]))
        latest_ckpt = os.path.join(model_dir, checkpoint_folders[-1])
    else:
        latest_ckpt = model_dir # Fallback if weights are in the root folder
        
    print(f"Loading model from: {latest_ckpt}")

    # 2. Pull processor config from base model
    ORIGINAL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"
    processor = Mask2FormerImageProcessor.from_pretrained(
        ORIGINAL_CHECKPOINT, 
        ignore_index=255, 
        reduce_labels=False
    )
    
    # 3. Load fine-tuned weights
    model = Mask2FormerForUniversalSegmentation.from_pretrained(latest_ckpt)
    model.to(device)
    model.eval()
    
    return processor, model

# =========================================================
# CLEAN OVERLAP RULES (UNTOUCHED)
# =========================================================

def clean_masks(masks):
    anode_wire = masks[1]
    cath_wire = masks[3]

    # Prevent the two wires from touching so the climber doesn't jump tracks
    overlap_wires = cath_wire & anode_wire
    cath_wire[overlap_wires] = False
    anode_wire[overlap_wires] = False

    return masks

# =========================================================
# DISTANCE (UNTOUCHED)
# =========================================================

def compute_distance_and_lines(masks): 
    mask_anode_wire = masks[1].astype(np.uint8) * 255     # Class 1
    mask_cath_wire = masks[3].astype(np.uint8) * 255      # Class 3

    distances_px = []
    traced_paths = [] 
    
    img_height, img_width = mask_anode_wire.shape

    # STEP 1: Find Absolute Top of Cathode Wires
    num_c, labels, stats, _ = cv2.connectedComponentsWithStats(mask_cath_wire)
    cathodes = []

    for i in range(1, num_c):
        x, y, w, h, area = stats[i]
        
        if area < 100 or y < img_height / 3: 
            continue
            
        local_ys, local_xs = np.where(labels == i)
        top_idx = np.argmin(local_ys)
        cx = local_xs[top_idx]
        cy = local_ys[top_idx]
        
        cathodes.append({'x': cx, 'y': cy})
        
    cathodes.sort(key=lambda c: c['x'])

    print("\n" + "="*50)
    print(f" 🛠️ METROLOGY: Tracing {len(cathodes)} Cathode Wires Left-to-Right")
    print("="*50)

    for i, cat in enumerate(cathodes):
        cx, cy = cat['x'], cat['y']

        # STEP 2: STRICT HORIZONTAL SCAN TO THE RIGHT
        start_x, start_y = None, None

        for x_step in range(cx, min(img_width, cx + 1500)):
            y_min_search = max(0, cy - 5)
            y_max_search = min(img_height, cy + 6)
            
            col = mask_anode_wire[y_min_search:y_max_search, x_step]
            
            if np.any(col > 0):
                local_ys = np.where(col > 0)[0]
                actual_ys = local_ys + y_min_search
                start_x = x_step
                start_y = actual_ys[np.argmin(np.abs(actual_ys - cy))]
                break

        if start_x is None:
            print(f"[{i+1}] FAILED: No Anode wire found straight to the right of ({cx}, {cy}).")
            continue

        curr_x, curr_y = start_x, start_y 
        path_distance = 0.0 
        current_path = [(cx, cy), (curr_x, curr_y)] 
        
        # STEP 3: STRICT FORWARD (UPWARD) CLIMBER
        while True:
            next_y = curr_y - 1
            if next_y < 0:
                break 
            
            x_min = max(0, curr_x - 3)
            x_max = min(img_width - 1, curr_x + 3)
            
            row_above = mask_anode_wire[next_y, x_min:x_max+1]
            
            if np.any(row_above > 0):
                xs = np.where(row_above > 0)[0]
                next_x = x_min + int(np.mean(xs))
                
                path_distance += math.hypot(next_x - curr_x, next_y - curr_y)
                curr_x = next_x
                curr_y = next_y
                current_path.append((curr_x, curr_y))
            else:
                gap_found = False
                for jump in range(2, 100): 
                    jump_y = curr_y - jump
                    if jump_y < 0: break
                    
                    jx_min = max(0, curr_x - 10)
                    jx_max = min(img_width - 1, curr_x + 10)
                    
                    gap_row = mask_anode_wire[jump_y, jx_min:jx_max+1]
                    if np.any(gap_row > 0):
                        xs = np.where(gap_row > 0)[0]
                        next_x = jx_min + int(np.mean(xs))
                        
                        path_distance += math.hypot(next_x - curr_x, jump_y - curr_y)
                        curr_x = next_x
                        curr_y = jump_y
                        current_path.append((curr_x, curr_y))
                        gap_found = True
                        break
                        
                if not gap_found:
                    break 

        # STEP 4: RECORD DISTANCE
        individual_um = path_distance * MICRONS_PER_PIXEL 
        distances_px.append(path_distance)
        traced_paths.append((current_path, individual_um)) 
        print(f"[{i+1}] SUCCESS: Reached Top of Anode Wire. Traced {individual_um:.2f}um.")

    if len(distances_px) == 0:
        return None, None

    avg_px = np.mean(distances_px)
    avg_um = avg_px * MICRONS_PER_PIXEL  

    return (avg_px, avg_um), traced_paths

# =========================================================
# MAIN
# =========================================================

def main():
    os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("Initializing...")
    processor, model = load_model(MODEL_DIR, device)

    img = cv2.imread(IMAGE_PATH)
    if img is None:
        raise FileNotFoundError(f"Could not load image at {IMAGE_PATH}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    # Initialize global masks using keys 1 (Anode) and 3 (Cathode)
    global_masks = { i: np.zeros((H, W), dtype=bool) for i in CLASS_MAPPING }

    # =====================================================
    # INFERENCE (SLIDING WINDOW OVER LARGE IMAGE)
    # =====================================================
    with torch.no_grad():
        for y in range(0, H, STRIDE):
            for x in range(0, W, STRIDE):
                y2 = min(y + CROP_SIZE, H)
                x2 = min(x + CROP_SIZE, W)

                crop = img[y:y2, x:x2]
                ch, cw = crop.shape[:2]

                # Pad edge crops to exactly 540x540 so the model behaves predictably
                padded = np.zeros((CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8)
                padded[:ch, :cw] = crop

                # Run through Mask2Former Image Processor
                inputs = processor(images=[padded], return_tensors="pt").to(device)
                outputs = model(**inputs)

                # Extract and combine logits (Mask2Former logic)
                class_logits = outputs.class_queries_logits
                mask_logits = outputs.masks_queries_logits

                class_probs = class_logits.softmax(dim=-1)[..., :-1]
                mask_probs = mask_logits.sigmoid()

                # Multiply class confidence by spatial mask probability
                semantic_probs = torch.einsum("bqc, bqhw -> bchw", class_probs, mask_probs)[0]

                # Resize the model's small output map precisely to 540x540
                semantic_probs = torch.nn.functional.interpolate(
                    semantic_probs.unsqueeze(0), 
                    size=(CROP_SIZE, CROP_SIZE), 
                    mode="bilinear", 
                    align_corners=False
                )[0]

                # Class 0 from model = Anode -> Mask Dict Key 1
                # Class 1 from model = Cathode -> Mask Dict Key 3
                pred_anode = (semantic_probs[0] > SCORE_THRESHOLD).cpu().numpy()
                pred_cathode = (semantic_probs[1] > SCORE_THRESHOLD).cpu().numpy()

                # Add to global mask (only mapping the valid [ch, cw] region to avoid padding bleed)
                global_masks[1][y:y2, x:x2] |= pred_anode[:ch, :cw]
                global_masks[3][y:y2, x:x2] |= pred_cathode[:ch, :cw]

    # =====================================================
    # RENDER SEGMENTATION (UNTOUCHED)
    # =====================================================
    vis = img.copy()

    for cid, info in sorted(CLASS_MAPPING.items(), key=lambda x: x[1]["order"]):
        m = global_masks[cid]
        if np.any(m):
            overlay = vis.copy()
            overlay[m] = info["color"]
            vis = cv2.addWeighted(overlay, info["alpha"], vis, 1 - info["alpha"], 0)

    # =====================================================
    # CLEAN MASKS & CALCULATE DISTANCE (UNTOUCHED)
    # =====================================================
    cleaned = clean_masks(global_masks)
    result, traced_paths = compute_distance_and_lines(cleaned)

    if result is not None:
        px, avg_um = result
        print(f"\nOVERALL Average Distance: {avg_um:.2f} µm")

        for path_data in traced_paths:
            path_coords = path_data[0]
            specific_um = path_data[1] 

            jump_pts = np.array(path_coords[:2], np.int32).reshape((-1, 1, 2))
            cv2.polylines(vis, [jump_pts], False, (0, 255, 255), 3, cv2.LINE_AA) 
            
            trace_pts = np.array(path_coords[1:], np.int32).reshape((-1, 1, 2))
            cv2.polylines(vis, [trace_pts], False, (255, 255, 255), 3, cv2.LINE_AA)

            top_x, top_y = path_coords[-1]
            cv2.putText(
                vis, f"{specific_um:.1f}um", (top_x + 15, top_y + 15),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3, cv2.LINE_AA
            )
    else:
        print("\n[WARNING] No distances could be calculated.")

    # =====================================================
    # SAVE FULL RESOLUTION
    # =====================================================
    out_path = os.path.join(OUTPUT_DIR, f"merged_{os.path.basename(IMAGE_PATH)}")
    cv2.imwrite(out_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    print("Saved:", out_path)

if __name__ == "__main__":
    main()

In [ ]:
pip install scikit-image

In [ ]:
import os
import cv2
import torch
import numpy as np
import math  
import matplotlib.pyplot as plt

# The brilliant suggestion: Skeletonization!
from skimage.morphology import skeletonize
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

# =========================================================
# CONFIG
# =========================================================

BASE_DIR = r"E:\Prithu\cropped_dataset"
MODEL_DIR = os.path.join(BASE_DIR, "mask2former_model")

INPUT_DIR = r"E:\Prithu\Anode\New folder"
OUTPUT_DIR = os.path.join(BASE_DIR, "predictions_large")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CROP_SIZE = 540
OVERLAP = 150
STRIDE = CROP_SIZE - OVERLAP

SCORE_THRESHOLD = 0.65 
MICRONS_PER_PIXEL = 1.72633

CLASS_MAPPING = {
    1: {"name": "Anode Wire", "color": (255, 0, 0), "alpha": 0.6, "order": 1},      # Red
    3: {"name": "Cathode Wire", "color": (0, 0, 255), "alpha": 0.6, "order": 2},    # Blue
}

# =========================================================
# MODEL
# =========================================================

def load_model(model_dir, device):
    checkpoint_folders = [f for f in os.listdir(model_dir) if f.startswith("checkpoint-")]
    if checkpoint_folders:
        checkpoint_folders.sort(key=lambda x: int(x.split("-")[-1]))
        latest_ckpt = os.path.join(model_dir, checkpoint_folders[-1])
    else:
        latest_ckpt = model_dir 
        
    print(f"Loading model from: {latest_ckpt}")

    ORIGINAL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"
    processor = Mask2FormerImageProcessor.from_pretrained(
        ORIGINAL_CHECKPOINT, 
        ignore_index=255, 
        reduce_labels=False
    )
    
    model = Mask2FormerForUniversalSegmentation.from_pretrained(latest_ckpt)
    model.to(device)
    model.eval()
    
    return processor, model

def clean_masks(masks):
    anode_wire = masks[1]
    cath_wire = masks[3]

    overlap_wires = cath_wire & anode_wire
    cath_wire[overlap_wires] = False
    anode_wire[overlap_wires] = False

    return masks

# =========================================================
# DISTANCE (SKELETON TRACER)
# =========================================================

def compute_distance_and_lines(masks): 
    mask_anode_wire = masks[1].astype(np.uint8) * 255     
    mask_cath_wire = masks[3].astype(np.uint8) * 255      

    distances_px = []
    traced_paths = [] 
    img_height, img_width = mask_anode_wire.shape

    # --- SKELETONIZATION ---
    # Reduce the thick Anode mask to a perfect 1-pixel center line spine
    bool_anode = mask_anode_wire > 0
    skel_anode = skeletonize(bool_anode)

    # STEP 1: Find Absolute Top of Cathode Wires
    num_c, labels_c, stats_c, _ = cv2.connectedComponentsWithStats(mask_cath_wire)
    cathodes = []

    for i in range(1, num_c):
        x, y, w, h, area = stats_c[i]
        
        # KILLS FALSE POSITIVES: Ignore anything smaller than 1000 pixels
        if area < 1000: 
            continue
            
        local_ys, local_xs = np.where(labels_c == i)
        top_idx = np.argmin(local_ys)
        cx = local_xs[top_idx]
        cy = local_ys[top_idx]
        
        cathodes.append({'x': cx, 'y': cy})
        
    cathodes.sort(key=lambda c: c['x'])

    print("\n" + "="*50)
    print(f" 🛠️ METROLOGY: Tracing {len(cathodes)} Cathode Wires Left-to-Right")
    print("="*50)

    for i, cat in enumerate(cathodes):
        cx, cy = cat['x'], cat['y']

        directions = [-1, 1] if i == 0 else [1]

        for d in directions:
            dir_name = "Left" if d == -1 else "Right"
            hit_x, hit_y = None, None

            # STEP 2: PERFECTLY STRAIGHT HORIZONTAL SCAN
            scan_range = range(cx, img_width) if d == 1 else range(cx, -1, -1)

            for x_step in scan_range:
                # ONLY checks the exact Y coordinate of the cathode top (perfectly straight line)
                if bool_anode[cy, x_step]:
                    hit_x, hit_y = x_step, cy
                    break

            if hit_x is None:
                print(f"[{i+1}] FAILED: No Anode wire found straight to the {dir_name} of ({cx}, {cy}).")
                continue
                
            # SNAP TO SKELETON: We hit the wall, now find the skeleton center to begin climbing
            skel_ys, skel_xs = np.where(skel_anode)
            if len(skel_ys) == 0:
                continue
                
            dists_to_hit = np.hypot(skel_xs - hit_x, skel_ys - hit_y)
            start_idx = np.argmin(dists_to_hit)
            start_sk_x, start_sk_y = skel_xs[start_idx], skel_ys[start_idx]

            curr_x, curr_y = start_sk_x, start_sk_y 
            path_distance = math.hypot(start_sk_x - hit_x, start_sk_y - hit_y)
            
            # Record the straight jump and the snap to the center
            current_path = [(cx, cy), (hit_x, hit_y), (curr_x, curr_y)] 
            
            # STEP 3: SKELETON TRACER
            visited = np.zeros((img_height, img_width), dtype=np.uint8)
            
            # Force the initial step UPWARD by blocking the area strictly below the start point
            cv2.rectangle(visited, (curr_x - 30, curr_y + 3), (curr_x + 30, img_height), 1, -1)
            
            brush_r = 5 # Small brush to erase the 1-pixel skeleton as we travel
            
            while True:
                # Erase traveled path
                cv2.circle(visited, (curr_x, curr_y), brush_r, 1, -1)
                
                # Look around for the next skeleton pixel
                r = 15
                y_min = max(0, curr_y - r)
                y_max = min(img_height - 1, curr_y + r)
                x_min = max(0, curr_x - r)
                x_max = min(img_width - 1, curr_x + r)
                
                window = skel_anode[y_min:y_max+1, x_min:x_max+1]
                v_window = visited[y_min:y_max+1, x_min:x_max+1] > 0
                
                valid = window & (~v_window)
                
                if np.any(valid):
                    ys, xs = np.where(valid)
                    ys += y_min
                    xs += x_min
                    
                    # Snap to the nearest unvisited skeleton pixel
                    dists = np.hypot(xs - curr_x, ys - curr_y)
                    min_idx = np.argmin(dists)
                    next_x, next_y = xs[min_idx], ys[min_idx]
                    
                    path_distance += dists[min_idx]
                    curr_x, curr_y = next_x, next_y
                    current_path.append((curr_x, curr_y))
                else:
                    break # Skeleton ended!

            # STEP 4: RECORD DISTANCE
            individual_um = path_distance * MICRONS_PER_PIXEL 
            distances_px.append(path_distance)
            traced_paths.append((current_path, individual_um)) 
            print(f"[{i+1}] SUCCESS: Reached Top of Anode Wire ({dir_name}). Traced {individual_um:.2f}um.")

    if len(distances_px) == 0:
        return None, None

    avg_px = np.mean(distances_px)
    avg_um = avg_px * MICRONS_PER_PIXEL  

    return (avg_px, avg_um), traced_paths

# =========================================================
# MAIN
# =========================================================

def main():
    os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("Initializing Model...")
    processor, model = load_model(MODEL_DIR, device)

    valid_extensions = ('.tif', '.tiff', '.png', '.jpg', '.jpeg')
    image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)]
    
    if not image_files:
        print(f"No valid images found in {INPUT_DIR}")
        return

    print(f"Found {len(image_files)} images to process.")

    for img_name in image_files:
        print(f"\n{'='*60}\nProcessing Image: {img_name}\n{'='*60}")
        img_path = os.path.join(INPUT_DIR, img_name)
        
        img = cv2.imread(img_path)
        if img is None:
            print(f"Error loading {img_name}, skipping...")
            continue
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        global_masks = { i: np.zeros((H, W), dtype=bool) for i in CLASS_MAPPING }

        # INFERENCE
        with torch.no_grad():
            for y in range(0, H, STRIDE):
                for x in range(0, W, STRIDE):
                    y2 = min(y + CROP_SIZE, H)
                    x2 = min(x + CROP_SIZE, W)

                    crop = img[y:y2, x:x2]
                    ch, cw = crop.shape[:2]

                    padded = np.zeros((CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8)
                    padded[:ch, :cw] = crop

                    inputs = processor(images=[padded], return_tensors="pt").to(device)
                    outputs = model(**inputs)

                    class_logits = outputs.class_queries_logits
                    mask_logits = outputs.masks_queries_logits

                    class_probs = class_logits.softmax(dim=-1)[..., :-1]
                    mask_probs = mask_logits.sigmoid()

                    semantic_probs = torch.einsum("bqc, bqhw -> bchw", class_probs, mask_probs)[0]

                    semantic_probs = torch.nn.functional.interpolate(
                        semantic_probs.unsqueeze(0), 
                        size=(CROP_SIZE, CROP_SIZE), 
                        mode="bilinear", 
                        align_corners=False
                    )[0]

                    pred_anode = (semantic_probs[0] > SCORE_THRESHOLD).cpu().numpy()
                    pred_cathode = (semantic_probs[1] > SCORE_THRESHOLD).cpu().numpy()

                    global_masks[1][y:y2, x:x2] |= pred_anode[:ch, :cw]
                    global_masks[3][y:y2, x:x2] |= pred_cathode[:ch, :cw]

        # RENDER SEGMENTATION
        vis = img.copy()
        for cid, info in sorted(CLASS_MAPPING.items(), key=lambda x: x[1]["order"]):
            m = global_masks[cid]
            if np.any(m):
                overlay = vis.copy()
                overlay[m] = info["color"]
                vis = cv2.addWeighted(overlay, info["alpha"], vis, 1 - info["alpha"], 0)

        # CLEAN MASKS & CALCULATE DISTANCE
        cleaned = clean_masks(global_masks)
        result, traced_paths = compute_distance_and_lines(cleaned)

        if result is not None:
            px, avg_um = result
            print(f"\nOVERALL Average Distance for {img_name}: {avg_um:.2f} µm")

            for path_data in traced_paths:
                path_coords = path_data[0]
                specific_um = path_data[1] 

                # THINNER VISUAL LINES (Thickness = 1 to see segmentation underneath)
                jump_pts = np.array(path_coords[:3], np.int32).reshape((-1, 1, 2))
                cv2.polylines(vis, [jump_pts], False, (0, 255, 255), 1, cv2.LINE_AA) 
                
                trace_pts = np.array(path_coords[2:], np.int32).reshape((-1, 1, 2))
                cv2.polylines(vis, [trace_pts], False, (255, 255, 255), 1, cv2.LINE_AA)

                top_x, top_y = path_coords[-1]
                cv2.putText(
                    vis, f"{specific_um:.1f}um", (top_x + 15, top_y + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA
                )
        else:
            print(f"\n[WARNING] No distances could be calculated for {img_name}.")

        # SAVE FULL RESOLUTION
        out_path = os.path.join(OUTPUT_DIR, f"merged_{img_name}")
        cv2.imwrite(out_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import torch
import cv2
import numpy as np
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from transformers import (
    Mask2FormerImageProcessor,
    Mask2FormerForUniversalSegmentation,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback
)

# =====================================================================
# 1. ENHANCED AUGMENTATIONS (Edge Sharpening + Morphological Contrast)
# =====================================================================
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    # CLAHE & Sharpen pop thin wire edges out from blurry backgrounds
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),
    A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.2, p=0.5),
    A.Affine(translate_percent=0.05, scale=(0.95, 1.05), rotate=15, p=0.5),
    A.ElasticTransform(p=0.2, alpha=1, sigma=50),
])

# =====================================================================
# 2. COCO DATASET LOADER (Polygon-Precise Rendering)
# =====================================================================
class COCOWireDataset(Dataset):
    def __init__(self, json_path, image_dir, processor, transforms=None):
        self.image_dir = image_dir
        self.processor = processor
        self.transforms = transforms

        with open(json_path, 'r') as f:
            self.coco = json.load(f)

        self.categories = self.coco.get('categories', [])
        self.cat_id_to_sem_id = {cat['id']: idx for idx, cat in enumerate(self.categories)}
        
        self.images = self.coco.get('images', [])
        self.img_to_anns = {}
        for ann in self.coco.get('annotations', []):
            self.img_to_anns.setdefault(ann['image_id'], []).append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']
        file_name = img_info['file_name']
        image_path = os.path.join(self.image_dir, file_name)

        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Image not found: {image_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        anns = self.img_to_anns.get(img_id, [])
        instance_mask = np.full((h, w), 255, dtype=np.int32)
        instance_id_to_semantic_id = {}

        for inst_idx, ann in enumerate(anns, start=1):
            if 'segmentation' in ann and ann['segmentation']:
                for poly in ann['segmentation']:
                    pts = np.array(poly, np.int32).reshape((-1, 2))
                    cv2.fillPoly(instance_mask, [pts], inst_idx)
            else:
                bbox = ann['bbox'] 
                x, y, bw, bh = [int(v) for v in bbox]
                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(w, x + bw), min(h, y + bh)
                instance_mask[y1:y2, x1:x2] = inst_idx

            cat_id = ann['category_id']
            instance_id_to_semantic_id[inst_idx] = self.cat_id_to_sem_id[cat_id]

        if self.transforms:
            augmented = self.transforms(image=image, mask=instance_mask)
            image = augmented['image']
            instance_mask = augmented['mask']
            # Revert Albumentations 0-padding back to ignore_index 255
            instance_mask[instance_mask == 0] = 255

        semantic_target = np.zeros((h, w), dtype=np.uint8)
        for inst_id, sem_id in instance_id_to_semantic_id.items():
            semantic_target[instance_mask == inst_id] = sem_id + 1

        return image, instance_mask, instance_id_to_semantic_id, semantic_target


def collate_fn(batch):
    images = [item[0] for item in batch]
    segmentation_maps = [item[1] for item in batch]
    instance_id_to_semantic_id = [item[2] for item in batch]

    inputs = processor(
        images=images,
        segmentation_maps=segmentation_maps,
        instance_id_to_semantic_id=instance_id_to_semantic_id,
        return_tensors="pt",
    )
    return inputs

# =====================================================================
# 3. BATCHED EVALUATION CALLBACK (Fast Epoch Validation)
# =====================================================================
class BatchedThresholdEvaluationCallback(TrainerCallback):
    def __init__(self, val_dataset, processor, anode_thresh=0.35, cathode_thresh=0.35, batch_size=16):
        self.val_dataset = val_dataset
        self.processor = processor
        self.anode_thresh = anode_thresh
        self.cathode_thresh = cathode_thresh
        self.batch_size = batch_size

    def on_evaluate(self, args, state, control, model, **kwargs):
        model.eval()
        device = args.device

        ious_anode, dices_anode = [], []
        ious_cathode, dices_cathode = [], []

        eval_loader = DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=lambda batch: (
                [item[0] for item in batch],
                [item[3] for item in batch]
            )
        )

        # FIX: Safe zero-division handling to prevent np.nan slice errors
        def calc_metrics(pred, target):
            inter = np.logical_and(pred, target).sum()
            union = np.logical_or(pred, target).sum()
            iou = inter / union if union > 0 else (1.0 if (pred.sum() == 0 and target.sum() == 0) else 0.0)
            dice = 2 * inter / (pred.sum() + target.sum()) if (pred.sum() + target.sum()) > 0 else (1.0 if (pred.sum() == 0 and target.sum() == 0) else 0.0)
            return iou, dice

        for images, semantic_targets in eval_loader:
            inputs = self.processor(images=images, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model(**inputs)
                class_logits = outputs.class_queries_logits
                mask_logits = outputs.masks_queries_logits

                class_probs = class_logits.softmax(dim=-1)[..., :-1]
                mask_probs = mask_logits.sigmoid()

                semantic_probs = torch.einsum("bqc, bqhw -> bchw", class_probs, mask_probs)

                target_h, target_w = semantic_targets[0].shape
                semantic_probs = torch.nn.functional.interpolate(
                    semantic_probs, 
                    size=(target_h, target_w), 
                    mode="bilinear", 
                    align_corners=False
                )

                pred_anodes = (semantic_probs[:, 0] > self.anode_thresh).cpu().numpy()
                pred_cathodes = (semantic_probs[:, 1] > self.cathode_thresh).cpu().numpy()

            for b in range(len(images)):
                target_anode = (semantic_targets[b] == 1)
                target_cathode = (semantic_targets[b] == 2)

                # SAFE SHAPE ALIGNMENT:
                # Find the guaranteed overlapping height/width between prediction and target
                ph, pw = pred_anodes[b].shape
                th, tw = target_anode.shape
                min_h = min(ph, th)
                min_w = min(pw, tw)
                
                # Slice BOTH arrays to the exact same valid image dimensions
                pred_a_cropped = pred_anodes[b][:min_h, :min_w]
                pred_c_cropped = pred_cathodes[b][:min_h, :min_w]
                target_a_cropped = target_anode[:min_h, :min_w]
                target_c_cropped = target_cathode[:min_h, :min_w]

                iou_a, dice_a = calc_metrics(pred_a_cropped, target_a_cropped)
                iou_c, dice_c = calc_metrics(pred_c_cropped, target_c_cropped)

                if not np.isnan(iou_a): ious_anode.append(iou_a)
                if not np.isnan(dice_a): dices_anode.append(dice_a)
                if not np.isnan(iou_c): ious_cathode.append(iou_c)
                if not np.isnan(dice_c): dices_cathode.append(dice_c)

        print(f"\n--- Epoch {state.epoch} Validation Metrics ---")
        print(f"Anode   (Thresh {self.anode_thresh:.2f}) | IoU: {np.mean(ious_anode):.4f} | Dice: {np.mean(dices_anode):.4f}")
        print(f"Cathode (Thresh {self.cathode_thresh:.2f}) | IoU: {np.mean(ious_cathode):.4f} | Dice: {np.mean(dices_cathode):.4f}")
        print("-------------------------------------------\n")

# =====================================================================
# 4. MAIN TRAINING EXECUTION
# =====================================================================
if __name__ == "__main__":
    BASE_DIR = r"E:\Prithu\cropped_dataset"
    JSON_PATH = os.path.join(BASE_DIR, "cropped_annotations.json")
    IMAGE_DIR = os.path.join(BASE_DIR, "images")
    
    # NEW DEDICATED MODEL FOLDER
    OUTPUT_MODEL_DIR = os.path.join(BASE_DIR, "mask2former_model_high_res_90iou")
    
    MODEL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"
    ID2LABEL = {0: "anode", 1: "cathode"}

    # Force native 540x540 resolution to prevent boundary blurring
    processor = Mask2FormerImageProcessor.from_pretrained(
        MODEL_CHECKPOINT, 
        ignore_index=255, 
        reduce_labels=False,
        do_resize=False, 
        do_pad=True,
        size={"height": 540, "width": 540}
    )
    model = Mask2FormerForUniversalSegmentation.from_pretrained(
        MODEL_CHECKPOINT, id2label=ID2LABEL, ignore_mismatched_sizes=True
    )

    full_dataset = COCOWireDataset(
        json_path=JSON_PATH, 
        image_dir=IMAGE_DIR, 
        processor=processor, 
        transforms=train_transform
    )

    val_size = int(0.2 * len(full_dataset))
    train_size = len(full_dataset) - val_size
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, val_size], generator=generator
    )

    # FIX: Smoother learning rates (2e-5) for full 540x540 resolution
    backbone_params = [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad]
    decoder_params = [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": 1e-5},
        {"params": decoder_params, "lr": 2e-5}, 
    ], weight_decay=0.05)

    training_args = TrainingArguments(
        output_dir=OUTPUT_MODEL_DIR,
        num_train_epochs=400,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        dataloader_num_workers=0,
        
        # --- NUMERICAL STABILITY & SCHEDULING ---
        fp16=False,                 # Disabled to prevent NaN loss
        bf16=True,                  # Stable 32-bit dynamic range on RTX A6000
        max_grad_norm=1.0,          # Clips gradient spikes
        lr_scheduler_type="cosine", # Smooth decay prevents query collapse
        warmup_ratio=0.05,          # 5% warmup stabilizes the Hungarian matcher
        
        remove_unused_columns=False,
    )

    early_stop = EarlyStoppingCallback(early_stopping_patience=50)
    custom_metrics = BatchedThresholdEvaluationCallback(
        val_dataset=val_dataset,
        processor=processor,
        anode_thresh=0.35,          # Optimal threshold for capturing thin edges
        cathode_thresh=0.35,
        batch_size=16
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate_fn,
        optimizers=(optimizer, None),
        callbacks=[early_stop, custom_metrics]
    )

    print(f"Loaded {len(full_dataset)} images from {BASE_DIR}")
    print(f"Saving high-resolution checkpoint to: {OUTPUT_MODEL_DIR}")
    print("Starting Training...")
    trainer.train()

In [ ]:
import os
import cv2
import torch
import numpy as np
import math  
import matplotlib.pyplot as plt

# The brilliant suggestion: Skeletonization!
from skimage.morphology import skeletonize
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

# =========================================================
# CONFIG
# =========================================================

BASE_DIR = r"E:\Prithu\cropped_dataset"
MODEL_DIR = os.path.join(BASE_DIR, "mask2former_model")

INPUT_DIR = r"E:\Prithu\Anode\New folder"
OUTPUT_DIR = os.path.join(BASE_DIR, "predictions_large")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CROP_SIZE = 540
OVERLAP = 150
STRIDE = CROP_SIZE - OVERLAP

SCORE_THRESHOLD = 0.65 
MICRONS_PER_PIXEL = 1.72633

CLASS_MAPPING = {
    1: {"name": "Anode Wire", "color": (255, 0, 0), "alpha": 0.6, "order": 1},      # Red
    3: {"name": "Cathode Wire", "color": (0, 0, 255), "alpha": 0.6, "order": 2},    # Blue
}

# =========================================================
# MODEL
# =========================================================

def load_model(model_dir, device):
    checkpoint_folders = [f for f in os.listdir(model_dir) if f.startswith("checkpoint-")]
    if checkpoint_folders:
        checkpoint_folders.sort(key=lambda x: int(x.split("-")[-1]))
        latest_ckpt = os.path.join(model_dir, checkpoint_folders[-1])
    else:
        latest_ckpt = model_dir 
        
    print(f"Loading model from: {latest_ckpt}")

    ORIGINAL_CHECKPOINT = "facebook/mask2former-swin-base-coco-instance"
    processor = Mask2FormerImageProcessor.from_pretrained(
        ORIGINAL_CHECKPOINT, 
        ignore_index=255, 
        reduce_labels=False
    )
    
    model = Mask2FormerForUniversalSegmentation.from_pretrained(latest_ckpt)
    model.to(device)
    model.eval()
    
    return processor, model

def clean_masks(masks):
    anode_wire = masks[1]
    cath_wire = masks[3]

    overlap_wires = cath_wire & anode_wire
    cath_wire[overlap_wires] = False
    anode_wire[overlap_wires] = False

    return masks

# =========================================================
# DISTANCE (90-DEGREE EXACT INTERSECTION TRACER)
# =========================================================

def compute_distance_and_lines(masks): 
    mask_anode_wire = masks[1].astype(np.uint8) * 255     
    mask_cath_wire = masks[3].astype(np.uint8) * 255      

    distances_px = []
    traced_paths = [] 
    img_height, img_width = mask_anode_wire.shape

    # --- 1. MORPHOLOGICAL PRE-SMOOTHING ---
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    smoothed_anode = cv2.morphologyEx(mask_anode_wire, cv2.MORPH_CLOSE, kernel)

    # --- 2. SKELETONIZATION ---
    bool_anode = smoothed_anode > 0
    skel_anode = skeletonize(bool_anode)

    # STEP 1: Find Absolute Top of Cathode Wires
    num_c, labels_c, stats_c, _ = cv2.connectedComponentsWithStats(mask_cath_wire)
    cathodes = []

    for i in range(1, num_c):
        x, y, w, h, area = stats_c[i]
        
        if area < 1000: 
            continue
            
        local_ys, local_xs = np.where(labels_c == i)
        top_idx = np.argmin(local_ys)
        cx = local_xs[top_idx]
        cy = local_ys[top_idx]
        
        cathodes.append({'x': cx, 'y': cy})
        
    cathodes.sort(key=lambda c: c['x'])

    print("\n" + "="*50)
    print(f" 🛠️ METROLOGY: Tracing {len(cathodes)} Cathode Wires Left-to-Right")
    print("="*50)

    for i, cat in enumerate(cathodes):
        cx, cy = cat['x'], cat['y']

        directions = [-1, 1] if i == 0 else [1]

        for d in directions:
            dir_name = "Left" if d == -1 else "Right"
            start_sk_x = None

            # STEP 2: STRICT 90-DEGREE HORIZONTAL SCAN TO FIND SKELETON CENTER
            # Instead of stopping at the outer wall and diagonally snapping,
            # we scan horizontally at exact Y (cy) until we hit the skeleton spine!
            scan_range = range(cx, img_width) if d == 1 else range(cx, -1, -1)

            for x_step in scan_range:
                if skel_anode[cy, x_step]:
                    start_sk_x = x_step
                    break
            
            # Fallback: If skeleton has a 1px gap at exact row cy, check cy-1 or cy+1
            if start_sk_x is None:
                for x_step in scan_range:
                    if (cy > 0 and skel_anode[cy - 1, x_step]) or (cy < img_height - 1 and skel_anode[cy + 1, x_step]):
                        start_sk_x = x_step
                        break

            if start_sk_x is None:
                print(f"[{i+1}] FAILED: No Anode skeleton found straight to the {dir_name} of ({cx}, {cy}).")
                continue

            # EXACT INTERSECTION POINT: (start_sk_x, cy)
            curr_x, curr_y = start_sk_x, cy 
            
            # Yellow/Cyan line goes strictly horizontal (90 deg) from (cx, cy) -> (start_sk_x, cy)
            # The white trace line starts at (start_sk_x, cy) and climbs UP.
            raw_path = [(start_sk_x, cy)] 
            
            # STEP 3: SKELETON TRACER
            visited = np.zeros((img_height, img_width), dtype=np.uint8)
            # Block everything strictly below the 90-degree intersection line so it ONLY climbs up
            cv2.rectangle(visited, (curr_x - 30, curr_y + 1), (curr_x + 30, img_height), 1, -1)
            
            brush_r = 5 
            
            while True:
                cv2.circle(visited, (curr_x, curr_y), brush_r, 1, -1)
                
                r = 15
                y_min = max(0, curr_y - r)
                y_max = min(img_height - 1, curr_y + r)
                x_min = max(0, curr_x - r)
                x_max = min(img_width - 1, curr_x + r)
                
                window = skel_anode[y_min:y_max+1, x_min:x_max+1]
                v_window = visited[y_min:y_max+1, x_min:x_max+1] > 0
                
                valid = window & (~v_window)
                
                if np.any(valid):
                    ys, xs = np.where(valid)
                    ys += y_min
                    xs += x_min
                    
                    dists = np.hypot(xs - curr_x, ys - curr_y)
                    min_idx = np.argmin(dists)
                    next_x, next_y = xs[min_idx], ys[min_idx]
                    
                    curr_x, curr_y = next_x, next_y
                    raw_path.append((curr_x, curr_y))
                else:
                    break # Skeleton ended!

            # --- 4. TRAJECTORY SMOOTHING (RDP ALGORITHM) ---
            pts_array = np.array(raw_path, dtype=np.float32).reshape(-1, 1, 2)
            smoothed_pts = cv2.approxPolyDP(pts_array, epsilon=1.8, closed=False)
            smoothed_path = [(int(p[0][0]), int(p[0][1])) for p in smoothed_pts]

            # Ensure the very first point of the smoothed path remains anchored to the exact 90-degree start
            smoothed_path[0] = (start_sk_x, cy)

            # Calculate true Euclidean distance along the smoothed trajectory
            path_distance = 0.0
            for pt_idx in range(len(smoothed_path) - 1):
                x1, y1 = smoothed_path[pt_idx]
                x2, y2 = smoothed_path[pt_idx + 1]
                path_distance += math.hypot(x2 - x1, y2 - y1)

            individual_um = path_distance * MICRONS_PER_PIXEL 
            distances_px.append(path_distance)
            
            # Store (Cathode_Start, 90_Deg_Intersection, Trace_Path, Distance)
            traced_paths.append(((cx, cy), smoothed_path, individual_um)) 
            print(f"[{i+1}] SUCCESS: Reached Top of Anode Wire ({dir_name}). Traced {individual_um:.2f}um.")

    if len(distances_px) == 0:
        return None, None

    avg_px = np.mean(distances_px)
    avg_um = avg_px * MICRONS_PER_PIXEL  

    return (avg_px, avg_um), traced_paths

# =========================================================
# MAIN
# =========================================================

def main():
    os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("Initializing Model...")
    processor, model = load_model(MODEL_DIR, device)

    valid_extensions = ('.tif', '.tiff', '.png', '.jpg', '.jpeg')
    image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)]
    
    if not image_files:
        print(f"No valid images found in {INPUT_DIR}")
        return

    print(f"Found {len(image_files)} images to process.")

    for img_name in image_files:
        print(f"\n{'='*60}\nProcessing Image: {img_name}\n{'='*60}")
        img_path = os.path.join(INPUT_DIR, img_name)
        
        img = cv2.imread(img_path)
        if img is None:
            print(f"Error loading {img_name}, skipping...")
            continue
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        global_masks = { i: np.zeros((H, W), dtype=bool) for i in CLASS_MAPPING }

        # INFERENCE
        with torch.no_grad():
            for y in range(0, H, STRIDE):
                for x in range(0, W, STRIDE):
                    y2 = min(y + CROP_SIZE, H)
                    x2 = min(x + CROP_SIZE, W)

                    crop = img[y:y2, x:x2]
                    ch, cw = crop.shape[:2]

                    padded = np.zeros((CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8)
                    padded[:ch, :cw] = crop

                    inputs = processor(images=[padded], return_tensors="pt").to(device)
                    outputs = model(**inputs)

                    class_logits = outputs.class_queries_logits
                    mask_logits = outputs.masks_queries_logits

                    class_probs = class_logits.softmax(dim=-1)[..., :-1]
                    mask_probs = mask_logits.sigmoid()

                    semantic_probs = torch.einsum("bqc, bqhw -> bchw", class_probs, mask_probs)[0]

                    semantic_probs = torch.nn.functional.interpolate(
                        semantic_probs.unsqueeze(0), 
                        size=(CROP_SIZE, CROP_SIZE), 
                        mode="bilinear", 
                        align_corners=False
                    )[0]

                    pred_anode = (semantic_probs[0] > SCORE_THRESHOLD).cpu().numpy()
                    pred_cathode = (semantic_probs[1] > SCORE_THRESHOLD).cpu().numpy()

                    global_masks[1][y:y2, x:x2] |= pred_anode[:ch, :cw]
                    global_masks[3][y:y2, x:x2] |= pred_cathode[:ch, :cw]

        # RENDER SEGMENTATION
        vis = img.copy()
        for cid, info in sorted(CLASS_MAPPING.items(), key=lambda x: x[1]["order"]):
            m = global_masks[cid]
            if np.any(m):
                overlay = vis.copy()
                overlay[m] = info["color"]
                vis = cv2.addWeighted(overlay, info["alpha"], vis, 1 - info["alpha"], 0)

        # CLEAN MASKS & CALCULATE DISTANCE
        cleaned = clean_masks(global_masks)
        result, traced_paths = compute_distance_and_lines(cleaned)

        if result is not None:
            px, avg_um = result
            print(f"\nOVERALL Average Distance for {img_name}: {avg_um:.2f} µm")

            for path_data in traced_paths:
                cath_start = path_data[0]       # (cx, cy)
                trace_coords = path_data[1]     # [(start_sk_x, cy), ...]
                specific_um = path_data[2] 

                # 1. DRAW PERFECT 90-DEGREE JUMP LINE (Yellow/Cyan)
                # Exactly horizontal from Cathode Top to Anode Skeleton Center
                jump_pts = np.array([cath_start, trace_coords[0]], np.int32).reshape((-1, 1, 2))
                cv2.polylines(vis, [jump_pts], False, (0, 255, 255), 1, cv2.LINE_AA) 
                
                # 2. DRAW ANODE TRACE LINE (White)
                # Starts strictly from trace_coords[0] (the exact 90-degree intersection)
                trace_pts = np.array(trace_coords, np.int32).reshape((-1, 1, 2))
                cv2.polylines(vis, [trace_pts], False, (255, 255, 255), 1, cv2.LINE_AA)

                top_x, top_y = trace_coords[-1]
                cv2.putText(
                    vis, f"{specific_um:.1f}um", (top_x + 15, top_y + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA
                )
        else:
            print(f"\n[WARNING] No distances could be calculated for {img_name}.")

        # SAVE FULL RESOLUTION
        out_path = os.path.join(OUTPUT_DIR, f"merged_{img_name}")
        cv2.imwrite(out_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()